In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import time
import math 

import networkx as nx

from coppeliasim_zmqremoteapi_client import RemoteAPIClient 

def Rz(theta):
  
    return np.array([[ np.cos(theta), -np.sin(theta), 0 ],
                      [ np.sin(theta), np.cos(theta) , 0 ],
                      [ 0            , 0             , 1 ]])

In [67]:
class HokuyoSensorSim(object):
    _sim = None
    _base_name = ""
    # _vision_sensor_name_template = "{}/fastHokuyo_joint{}"
    _vision_sensor_name_template = "{}/fastHokuyo_joint{}/fastHokuyo_sensor{}"
    _base_obj = None
    _is_range_data = False
    
    _angle_min = -120*math.pi/180
    _angle_max = 120*math.pi/180
    _angle_increment = (240/684)*math.pi/180  # angle: 240 deg, pts: 684

    def __init__(self, sim, base_name, is_range_data=True):
        self._sim = sim
        self._base_name = base_name
        self._is_range_data = is_range_data

        if "fastHokuyo" not in base_name:
            raise ValueError("ERR: fastHokuyo must be in the base object name.")

        self._base_obj = sim.getObject(base_name)
        if self._base_obj == -1:
            raise ValueError("ERR: base_obj is not a valid name in the simulation")


        print(f"{self._vision_sensor_name_template.format(self._base_name, 1, 1)}")
        print(f"{self._vision_sensor_name_template.format(self._base_name, 2, 2)}")
        
        # self._vision_sensor_name_template = "{}/fastHokuyo/fastHokuyo_body/fastHokuyo_joint{}/fastHokuyo_sensor{}"
        # self._vision_sensor_name_template = "{}/fastHokuyo/fastHokuyo_joint{}/fastHokuyo_sensor{}"

        # self._vision_sensors_obj = [
        #     sim.getObject(self._vision_sensor_name_template.format(self._base_name, 1, 1)),
        #     sim.getObject(self._vision_sensor_name_template.format(self._base_name, 2, 2)),
        # ]

        self._vision_sensors_obj = [
            sim.getObject(self._vision_sensor_name_template.format(self._base_name, 1,1)),
            sim.getObject(self._vision_sensor_name_template.format(self._base_name, 2,2)),
        ]

        if any(obj == -1 for obj in self._vision_sensors_obj):
            raise ValueError("ERR: the vision sensors are not valid in the simulation")

    def getSensorData(self):
        angle = self._angle_min
        sensor_data = []
        
        for vision_sensor in self._vision_sensors_obj:
            r, t, u = sim.readVisionSensor(vision_sensor)
            if u:
                for j in range(int(u[1])):
                    for k in range(int(u[0])):
                        w = 2 + 4 * (j * int(u[0]) + k)
                        v = [u[w], u[w + 1], u[w + 2], u[w + 3]]
                        angle = angle + self._angle_increment
                        if self._is_range_data:
                            sensor_data.append([angle, v[3]])
                        else:
                            sensor_data.append([v[0], v[1], v[2]])
                            
        return np.array(sensor_data)


def transform_laser_to_global(laser_data, robot_pos, robot_ori):

    x_r, y_r = robot_pos[0], robot_pos[1]
    theta_r = robot_ori[2]  # orientação em torno de z

    global_points = []

    for ang, dist in laser_data:
        if dist > 0.01 and dist < 5:
            # coordenadas no robô
            x_local = dist * np.cos(ang)
            y_local = dist * np.sin(ang)

            # transformação para global
            x_global = x_r + x_local*np.cos(theta_r) - y_local*np.sin(theta_r)
            y_global = y_r + x_local*np.sin(theta_r) + y_local*np.cos(theta_r)

            global_points.append([x_global, y_global])

    return global_points

In [4]:
def rep_force(q, obs_array, R, krep, d_min, max_f_rep):
    if obs_array is None or len(obs_array) == 0:
        return np.zeros(2)

    q = np.asarray(q).reshape(1, -1)
    obs = np.asarray(obs_array).reshape(-1, 2)
    v = q - obs
    d = np.linalg.norm(v, axis=1)
    d_safe = np.maximum(d, d_min)

    within = d < R
    if not np.any(within):
        return np.zeros(2)

    coeff = np.zeros_like(d_safe)
    coeff[within] = krep * ((1.0 / d_safe[within]) - (1.0 / R)) * (1.0 / (d_safe[within] ** 2))
    dirs = v / d_safe.reshape(-1, 1)
    rep = (coeff.reshape(-1, 1) * dirs)
    force_sum = np.sum(rep, axis=0)

    idx_closest = np.argmin(d_safe)
    dir_closest = dirs[idx_closest]
    tangential = np.array([-dir_closest[1], dir_closest[0]])
    tangential_gain = 0.4
    tangential_component = tangential_gain * tangential * min(1.0, np.linalg.norm(force_sum) / (max_f_rep + 1e-6))
    force_sum = force_sum + tangential_component

    norm_f = np.linalg.norm(force_sum)
    if norm_f > max_f_rep:
        force_sum = (force_sum / norm_f) * max_f_rep

    return force_sum


def att_force(q, goal, k, max_f_att):
    f = k * (np.asarray(goal) - np.asarray(q))
    norm = np.linalg.norm(f)
    if norm > max_f_att:
        f = f / norm * max_f_att
    return f

# validacao dos pontos dos sensores
def filter_valid_points(points, robot_pos, min_dist, max_dist):
    if points is None or len(points) == 0:
        return np.empty((0, 2))
    pts = np.asarray(points).reshape(-1, 2)
    d = np.linalg.norm(pts - robot_pos.reshape(1, 2), axis=1)
    mask = np.isfinite(d) & (d > min_dist) & (d < max_dist)
    return pts[mask]


def transform_laser_to_global(laser_points, posRobo_full, oriRobo):
    if laser_points is None or len(laser_points) == 0:
        return np.empty((0, 2))

    x_r, y_r, _ = posRobo_full
    theta = oriRobo[2]

    R = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])
    pts = np.asarray(laser_points).reshape(-1, 2)
    pts_global = (R @ pts.T).T + np.array([x_r, y_r])
    return pts_global




def escape_if_stuck(robot_pos, prev_pos, stuck_timer, escape_mode, last_escape_time, sim_time, dt):
    stuck_threshold = 0.02
    stuck_time_limit = 1.2
    escape_duration = 2.0
    min_time_between_escapes = 3.0

    moved_dist = np.linalg.norm(robot_pos - prev_pos)
    if moved_dist < stuck_threshold:
        stuck_timer += dt
    else:
        stuck_timer = 0.0

    if (stuck_timer > stuck_time_limit) and (not escape_mode) and ((sim_time - last_escape_time) > min_time_between_escapes):
        escape_mode = True
        last_escape_time = sim_time
        print("[ESCAPE] Robo preso — iniciando rotina de escape")

    if escape_mode:
        if (sim_time - last_escape_time) < escape_duration:
            v = -0.15
            bias = 1.0 if ((int(last_escape_time) % 2) == 0) else -1.0
            w = bias * 0.9
        else:
            escape_mode = False
            v, w = None, None
            stuck_timer = 0.0
    else:
        v, w = None, None

    return v, w, stuck_timer, escape_mode, last_escape_time



def pixel_to_world(x_px, y_px, img_size=64, world_size=10):
        scale = world_size / img_size
        x_world = ((x_px - img_size/2) * scale)
        y_world = (-(y_px - img_size/2) * scale) 
        return x_world, y_world



In [68]:
print('Program started')

# constantes
ATT_GAIN = 1.0
REP_GAIN = 80.0
REP_RADIUS = 0.8
MAX_REP_FORCE = 25.0

LINEAR_GAIN = 0.25
ANGULAR_GAIN = 2.0

MAX_LINEAR_VEL = 0.8
MAX_ANGULAR_VEL = np.deg2rad(90)

GOAL_TOL = 0.20

prev_v_cmd = 0.0
v_smoothing = 0.12


# simulacao

client = RemoteAPIClient()
sim = client.require("sim")

sim.setStepping(True)


#Configuracao robo kobuki

L = 0.230  # distancia do centro as rodas (m)
r = 0.035  # raio das rodas (m)

robotname = "kobuki"

robotHandle = sim.getObject(f'/{robotname}')
l_wheel = sim.getObject(f'/{robotname}/wheel_left_drop_sensor/kobuki_leftMotor')
r_wheel = sim.getObject(f'/{robotname}/wheel_right_drop_sensor/kobuki_rightMotor')

# l_wheel = sim.getObject(f'/{robotname}/kuboki_leftMotor')
# r_wheel = sim.getObject(f'/{robotname}/kuboki_rightMotor')

world = pixel_to_world(89,54,100,10)
goal_position = np.array([world[0], world[1]])


sim.startSimulation()

hokuyo_sensor = HokuyoSensorSim(sim, "/kobuki/fastHokuyo")
has_sensor = True



    
prev_pos = np.array(sim.getObjectPosition(robotHandle, sim.handle_world)[:2])
stuck_timer = 0.0
escape_mode = False
last_escape_time = -999.0
last_sim_time = sim.getSimulationTime()

while True:
    sim_time = sim.getSimulationTime()
    dt = sim_time - last_sim_time if sim_time - last_sim_time > 0 else 0.05
    last_sim_time = sim_time

    posRobo_full = sim.getObjectPosition(robotHandle, sim.handle_world)
    oriRobo = sim.getObjectOrientation(robotHandle, sim.handle_world)
    robot_pos = np.array([posRobo_full[0], posRobo_full[1]])

    if np.linalg.norm(goal_position - robot_pos) <= GOAL_TOL:
        print("Objetivo alcancado.")
        break

    v_escape, w_escape, stuck_timer, escape_mode, last_escape_time = escape_if_stuck(
        robot_pos, prev_pos, stuck_timer, escape_mode, last_escape_time, sim_time, dt
    )

    if escape_mode and (v_escape is not None):
        v_cmd = v_escape
        w_cmd = w_escape
    else:
        if has_sensor:
            laser_data = hokuyo_sensor.getSensorData()
        #     laser_global = transform_laser_to_global(laser_data, posRobo_full, oriRobo)
        #     laser_global = filter_valid_points(laser_global, robot_pos, min_dist=0.05, max_dist= 0.05)
        # else:
        #     laser_global = np.empty((0, 2))

    #     f_att = att_force(robot_pos, goal_position, k=ATT_GAIN, max_f_att=20.0)
    #     f_rep = rep_force(robot_pos, laser_global, R=REP_RADIUS, krep=REP_GAIN, d_min=0.03, max_f_rep=MAX_REP_FORCE)

    #     f_total = f_att + f_rep
    #     if np.linalg.norm(f_total) < 1e-3:
    #         jitter = 0.05 * (np.random.rand(2) - 0.5)
    #         f_total = f_total + jitter

    
    #     # CONTROLADOR DESAI ET AL. (1998)
    #     xd, yd = f_total
    #     theta = oriRobo[2]
    #     d = L / 2.0  

    #     J = np.array([
    #         [np.cos(theta), np.sin(theta)],
    #         [-np.sin(theta) / d, np.cos(theta) / d]
    #     ])

    #     v_cmd, w_cmd = J @ np.array([xd, yd])

    #     v_cmd *= LINEAR_GAIN
    #     w_cmd *= ANGULAR_GAIN

    #     theta_d = np.arctan2(f_total[1], f_total[0])
    #     erro_theta = np.arctan2(np.sin(theta_d - theta), np.cos(theta_d - theta))
    #     if abs(erro_theta) > np.deg2rad(45):
    #         v_cmd *= 0.25

    #     v_cmd = np.clip(v_cmd, -MAX_LINEAR_VEL, MAX_LINEAR_VEL)
    #     w_cmd = np.clip(w_cmd, -MAX_ANGULAR_VEL, MAX_ANGULAR_VEL)

    #     prev_v_cmd = v_cmd

    # # cinematica diferencial
    # v_r = (2.0 * v_cmd + w_cmd * L) / (2.0 * r)
    # v_l = (2.0 * v_cmd - w_cmd * L) / (2.0 * r)

    # sim.setJointTargetVelocity(l_wheel, v_l)
    # sim.setJointTargetVelocity(r_wheel, v_r)

    # print(f"t={sim_time:.2f} | pos={robot_pos} | v={v_cmd:.3f}, w={np.rad2deg(w_cmd):.1f}°/s | escape={escape_mode} | stuck_t={stuck_timer:.2f}")

    # prev_pos = robot_pos.copy()
    sim.step()

sim.setJointTargetVelocity(l_wheel, 0)
sim.setJointTargetVelocity(r_wheel, 0)

sim.stopSimulation()
print('Simulação finalizada.')

Program started


/kobuki/fastHokuyo/fastHokuyo_joint1/fastHokuyo_sensor1
/kobuki/fastHokuyo/fastHokuyo_joint2/fastHokuyo_sensor2


TypeError: cannot unpack non-iterable int object

In [5]:


# Conexao Coppelia








world = pixel_to_world(89,54,100,10)
goal_position = np.array([world[0], world[1]])



# Informacao do robotino
L = 0.135  # distancia do centro as rodas (m)
r = 0.040  # raio das rodas (m)


# pos_world = {node: pixel_to_world(x_px, y_px, 100, 10) for node, (x_px, y_px) in pos.items()}

# Mdir = np.array([[-r/np.sqrt(3),     0,        r/np.sqrt(3)], 
#                  [r/3,            (-2*r)/3,    r/3], 
#                  [r/(3*L),         r/(3*L),    r/(3*L)]])


# # Posicao inicial do robo - o mesmo da caminho
# start_node = path[0]
# start_pos_world = pos_world[start_node]
# start_pos_sim = [start_pos_world[0], start_pos_world[1], 0.08]
# start_ori_sim = [0, 0, np.deg2rad(90)]




# # Parar a simulacao se estiver executando
# initial_sim_state = sim.getSimulationState()
# if initial_sim_state != 0:
#     sim.stopSimulation()
#     time.sleep(1)

# sim.startSimulation()
# sim.setObjectPosition(robotHandle, sim.handle_world, start_pos_sim)
# sim.setObjectOrientation(robotHandle, sim.handle_world, start_ori_sim)
# sim.step()





# sim.setStepping(True)
# sim.stopSimulation()
# time.sleep(0.5)




# L = 0.381
# r = 0.0975


# hokuyo_sensor = HokuyoSensorSim(sim, "/PioneerP3DX/fastHokuyo")
# has_sensor = True


# sim.startSimulation()

# prev_pos = np.array(sim.getObjectPosition(robotHandle, sim.handle_world)[:2])
# stuck_timer = 0.0
# escape_mode = False
# last_escape_time = -999.0
# last_sim_time = sim.getSimulationTime()

# while True:
#     sim_time = sim.getSimulationTime()
#     dt = sim_time - last_sim_time if sim_time - last_sim_time > 0 else 0.05
#     last_sim_time = sim_time

#     posRobo_full = sim.getObjectPosition(robotHandle, sim.handle_world)
#     oriRobo = sim.getObjectOrientation(robotHandle, sim.handle_world)
#     robot_pos = np.array([posRobo_full[0], posRobo_full[1]])

#     if np.linalg.norm(goal_position - robot_pos) <= GOAL_TOL:
#         print("Objetivo alcancado.")
#         break

#     v_escape, w_escape, stuck_timer, escape_mode, last_escape_time = escape_if_stuck(
#         robot_pos, prev_pos, stuck_timer, escape_mode, last_escape_time, sim_time, dt
#     )

#     if escape_mode and (v_escape is not None):
#         v_cmd = v_escape
#         w_cmd = w_escape
#     else:
#         if has_sensor:
#             laser_data = hokuyo_sensor.getSensorData()
#             laser_global = transform_laser_to_global(laser_data, posRobo_full, oriRobo)
#             laser_global = filter_valid_points(laser_global, robot_pos, min_dist=0.05, max_dist= 0.05)
#         else:
#             laser_global = np.empty((0, 2))

#         f_att = att_force(robot_pos, goal_position, k=ATT_GAIN, max_f_att=20.0)
#         f_rep = rep_force(robot_pos, laser_global, R=REP_RADIUS, krep=REP_GAIN, d_min=0.03, max_f_rep=MAX_REP_FORCE)

#         f_total = f_att + f_rep
#         if np.linalg.norm(f_total) < 1e-3:
#             jitter = 0.05 * (np.random.rand(2) - 0.5)
#             f_total = f_total + jitter

    
#         # CONTROLADOR DESAI ET AL. (1998)
#         xd, yd = f_total
#         theta = oriRobo[2]
#         d = L / 2.0  

#         J = np.array([
#             [np.cos(theta), np.sin(theta)],
#             [-np.sin(theta) / d, np.cos(theta) / d]
#         ])

#         v_cmd, w_cmd = J @ np.array([xd, yd])

#         v_cmd *= LINEAR_GAIN
#         w_cmd *= ANGULAR_GAIN

#         theta_d = np.arctan2(f_total[1], f_total[0])
#         erro_theta = np.arctan2(np.sin(theta_d - theta), np.cos(theta_d - theta))
#         if abs(erro_theta) > np.deg2rad(45):
#             v_cmd *= 0.25

#         v_cmd = np.clip(v_cmd, -MAX_LINEAR_VEL, MAX_LINEAR_VEL)
#         w_cmd = np.clip(w_cmd, -MAX_ANGULAR_VEL, MAX_ANGULAR_VEL)

#         prev_v_cmd = v_cmd

#     # cinematica diferencial
#     v_r = (2.0 * v_cmd + w_cmd * L) / (2.0 * r)
#     v_l = (2.0 * v_cmd - w_cmd * L) / (2.0 * r)

#     sim.setJointTargetVelocity(l_wheel, v_l)
#     sim.setJointTargetVelocity(r_wheel, v_r)

#     print(f"t={sim_time:.2f} | pos={robot_pos} | v={v_cmd:.3f}, w={np.rad2deg(w_cmd):.1f}°/s | escape={escape_mode} | stuck_t={stuck_timer:.2f}")

#     prev_pos = robot_pos.copy()
#     sim.step()

# sim.setJointTargetVelocity(l_wheel, 0)
# sim.setJointTargetVelocity(r_wheel, 0)

sim.stopSimulation()
print('Simulação finalizada.')
    
    

Program started


NameError: name 'pos' is not defined